In [3]:
!pip install -q transformers datasets peft trl accelerate
!pip install -q "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 3.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 52.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

In [4]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print(dataset)
print("\n--- Sample entry ---")
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

--- Sample entry ---
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [5]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Qwen3 doesn't always set a pad token — fix that
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Vocab size: 151643
Pad token: <|endoftext|>


In [6]:
def format_example(example):
    """
    GSM8K fields:
      - example["question"]: the math word problem
      - example["answer"]: solution with chain-of-thought + #### final_answer
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful math tutor. Solve the problem step by step, then give the final answer after ####."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]
    
    # Apply Qwen3's built-in chat template
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {"text": formatted}

# Apply to both splits
train_dataset = dataset["train"].map(format_example)
test_dataset  = dataset["test"].map(format_example)

print("Train size:", len(train_dataset))
print("Test size: ", len(test_dataset))
print("\n--- Formatted example ---")
print(train_dataset[0]["text"])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train size: 7473
Test size:  1319

--- Formatted example ---
<|im_start|>system
You are a helpful math tutor. Solve the problem step by step, then give the final answer after ####.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
<think>

</think>

Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [7]:
train_subset = train_dataset
test_subset  = test_dataset

print("Train size:", len(train_subset), "| Test:", len(test_subset))

Train size: 7473 | Test: 1319


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3-0.6B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
print("Model loaded!")

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded!


In [9]:
trials = [
    {
        "trial_id": 1,
        "lora_r": 8,
        "lora_alpha": 16,
        "target_modules": ["q_proj", "v_proj"],
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 4,
        "num_train_epochs": 2,
    },
    {
        "trial_id": 2,
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "v_proj"],
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 4,
        "num_train_epochs": 3,
    },
    {
        "trial_id": 3,
        "lora_r": 32,
        "lora_alpha": 64,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "learning_rate": 1e-4,
        "per_device_train_batch_size": 8,
        "num_train_epochs": 2,
    },
    {
        "trial_id": 4,
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "learning_rate": 5e-5,
        "per_device_train_batch_size": 4,
        "num_train_epochs": 3,
    },
    {
        "trial_id": 5,
        "lora_r": 64,
        "lora_alpha": 128,
        "target_modules": ["q_proj", "v_proj"],
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 8,
        "num_train_epochs": 1,
    },
]

print(f"{len(trials)} trials defined.")

5 trials defined.


In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling

# Only run remaining trials
trials_remaining = [t for t in trials if t['trial_id'] in [3, 4, 5]]

# Manually add already-completed results
trial_results = [
    {"trial_id": 1, "train_loss": 3.1166, "eval_loss": float('nan'), "epochs": 2},
    {"trial_id": 2, "train_loss": 3.8347, "eval_loss": float('nan'), "epochs": 3},
]

use_bf16 = torch.cuda.is_bf16_supported()

for trial in trials_remaining:
    print(f"\n{'='*50}")
    print(f"  TRIAL {trial['trial_id']}")
    print(f"{'='*50}")

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map={"": 0},
            trust_remote_code=True
        )
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

        lora_config = LoraConfig(
            r=trial["lora_r"],
            lora_alpha=trial["lora_alpha"],
            target_modules=trial["target_modules"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

        output_dir = f"/kaggle/working/sft_trial_{trial['trial_id']}"

        sft_config = SFTConfig(
            output_dir=output_dir,
            num_train_epochs=trial["num_train_epochs"],
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            learning_rate=trial["learning_rate"],
            bf16=use_bf16,
            fp16=not use_bf16,
            optim="paged_adamw_8bit",
            logging_steps=50,
            eval_strategy="epoch",
            eval_accumulation_steps=4,
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=False,
            report_to="none",
            max_length=512,
            dataset_text_field="text",
            dataset_kwargs={"add_special_tokens": False},
            packing=False,
            ddp_find_unused_parameters=False,
        )

        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        trainer = SFTTrainer(
            model=model,
            args=sft_config,
            train_dataset=train_subset,
            eval_dataset=test_subset,
            processing_class=tokenizer,
            data_collator=data_collator,
        )

        train_result = trainer.train()
        eval_result  = trainer.evaluate()

        trial_results.append({
            "trial_id":       trial["trial_id"],
            "lora_r":         trial["lora_r"],
            "target_modules": str(trial["target_modules"]),
            "lr":             trial["learning_rate"],
            "batch_size":     trial["per_device_train_batch_size"],
            "epochs":         trial["num_train_epochs"],
            "train_loss":     round(train_result.training_loss, 4),
            "eval_loss":      round(eval_result["eval_loss"], 4),
        })

        trainer.save_model(output_dir)
        print(f"Trial {trial['trial_id']} saved to {output_dir}")

    except Exception as e:
        print(f"Trial {trial['trial_id']} FAILED: {e}")
        trial_results.append({"trial_id": trial["trial_id"], "error": str(e)})

    finally:
        try: del model
        except NameError: pass
        try: del trainer
        except NameError: pass
        gc.collect()
        torch.cuda.empty_cache()

print("\nAll trials complete!")
print(trial_results)


  TRIAL 3


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 9,175,040 || all params: 760,807,424 || trainable%: 1.2060


Adding EOS to train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 9,175,040 || all params: 760,807,424 || trainable%: 1.2060


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.708260,0.794199,0.750593,1669771.000000,0.803507


Trial 5 saved to /kaggle/working/sft_trial_5

All trials complete!
[{'trial_id': 1, 'train_loss': 3.1166, 'eval_loss': nan, 'epochs': 2}, {'trial_id': 2, 'train_loss': 3.8347, 'eval_loss': nan, 'epochs': 3}, {'trial_id': 3, 'lora_r': 32, 'target_modules': "['q_proj', 'k_proj', 'v_proj', 'o_proj']", 'lr': 0.0001, 'batch_size': 8, 'epochs': 2, 'train_loss': 0.7129, 'eval_loss': 0.8035}, {'trial_id': 4, 'lora_r': 16, 'target_modules': "['q_proj', 'k_proj', 'v_proj', 'o_proj']", 'lr': 5e-05, 'batch_size': 4, 'epochs': 3, 'train_loss': 0.7443, 'eval_loss': 0.8198}, {'trial_id': 5, 'lora_r': 64, 'target_modules': "['q_proj', 'v_proj']", 'lr': 0.0002, 'batch_size': 8, 'epochs': 1, 'train_loss': 0.7516, 'eval_loss': 0.7942}]


In [12]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_subset,
    eval_dataset=test_subset,
    processing_class=tokenizer,
    data_collator=data_collator,   # ← add this
)

In [13]:
def fix_text(example):
    # Remove empty think blocks
    example["text"] = example["text"].replace("<think>\n\n</think>\n\n", "")
    return example

train_subset = train_subset.map(fix_text)
test_subset  = test_subset.map(fix_text)

# Verify
print(train_subset[0]["text"])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful math tutor. Solve the problem step by step, then give the final answer after ####.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [11]:
import shutil
shutil.make_archive("/kaggle/working/all_trials", "zip", "/kaggle/working")

'/kaggle/working/all_trials.zip'

In [12]:
from IPython.display import FileLink
FileLink('/kaggle/working/all_trials.zip')

/kaggle/working/all_trials.zip

In [18]:
import subprocess, shutil

for i in [3, 4, 5]:
    # Copy with .bin extension
    shutil.copy(
        f'/kaggle/working/sft_trial_{i}/adapter_model.safetensors',
        f'/kaggle/working/sft_trial_{i}/adapter_model.bin'
    )
    r = subprocess.run(
        ['curl', '-F', f'file=@/kaggle/working/sft_trial_{i}/adapter_model.bin',
         'https://tmpfiles.org/api/v1/upload'],
        capture_output=True, text=True
    )
    print(f"Trial {i}:", r.stdout)

Trial 3: {"status":"success","data":{"url":"https://tmpfiles.org/wGwgeErFxMBX/adapter_model.bin"}}
Trial 4: {"status":"success","data":{"url":"https://tmpfiles.org/wOwCeFr2xhh7/adapter_model.bin"}}
Trial 5: {"status":"success","data":{"url":"https://tmpfiles.org/wPwielrOxdEf/adapter_model.bin"}}


In [19]:
import subprocess

for i in [3, 4, 5]:
    r = subprocess.run(
        ['curl', '--upload-file', 
         f'/kaggle/working/sft_trial_{i}/adapter_model.safetensors',
         f'https://transfer.sh/adapter_model_trial_{i}.safetensors'],
        capture_output=True, text=True
    )
    print(f"Trial {i}:", r.stdout)

Trial 3: 
Trial 4: 
Trial 5: 
